In [17]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from nsdget import nsd_betas_images_trials, nsd_coco_image, nsd_single_trial_betas, nsd_betas_nii_to_numpy, NiiImage
from nilearn.image import resample_img
import nibabel as nb
from nilearn.datasets import load_mni152_template

# download and use data
df: pd.DataFrame = nsd_betas_images_trials(save_to="../nsdata/")
df.head()

> Data in Data points to ../nsdata/
>> Downloading All Betas


100%|██████████| 40/40 [00:00<00:00, 128168.19it/s]


100%|██████████| 40/40 [00:00<00:00, 164643.93it/s]


100%|██████████| 32/32 [00:00<00:00, 176602.27it/s]


100%|██████████| 30/30 [00:00<00:00, 64560.86it/s]


...

100%|██████████| 30/30 [00:00<00:00, 86958.62it/s]

>> Downloading 73k COCO Images



100%|██████████| 73000/73000 [00:00<00:00, 170987.38it/s]


>> Unrolling stimulu info into dataframe


,subjectId,trialId,cocoId,cocoSplit,cropBox,loss,nsdId,flagged,BOLD5000,shared1000,sessionId,sessionTrialId
0,1,1,412922,train2017,"[0.0, 0.0, 0.125, 0.125]",0.071429,46002,False,True,True,1,1
1,1,2,474858,train2017,"[0.0, 0.0, 0.16640625, 0.16640625]",0.130435,61882,False,False,False,1,1
2,1,3,320696,val2017,"[0.0, 0.0, 0.16640625, 0.16640625]",0.000000,828,False,False,False,1,1
3,1,4,234676,train2017,"[0.0, 0.0, 0.16640625, 0.16640625]",0.000000,67573,False,False,False,1,1
4,1,5,301595,train2017,"[0.125, 0.125, 0.0, 0.0]",0.000000,16020,False,False,False,1,1


In [20]:
class NSDTrials(Dataset):
    def __init__(self, df, transforms=[]):
        self.df = df
        self.transforms = transforms
    def __getitem__(self, index):
        row = self.df.iloc[index]
        betas = nsd_single_trial_betas(row, numpy=False)
        for t in self.transforms:
            betas = t(betas)
        image = nsd_coco_image(row)
        return betas

    def __len__(self):
        return len(self.df)

# Load the MNI template (default: 2mm resolution)
mni_template = load_mni152_template(resolution=2)
mni_affine = mni_template.affine  # Target affine (4x4 matrix)
mni_shape = mni_template.shape    # Target shape (3D dimensions)

def transform_native_to_mni152_2mm(img: NiiImage) -> NiiImage:
    global mni_affine, mni_shape
    return resample_img(
        img=img,        # Your subject-native beta map
        target_affine=mni_affine,        # MNI template affine
        target_shape=mni_shape,          # MNI template shape (optional but recommended)
        interpolation='continuous',      # Cubic interpolation (matches NSD's method)
        fill_value=0,                    # Set non-brain voxels to 0 (like NSD)
        clip=True                        # Clip values outside original range
    )

ds = NSDTrials(df, transforms=[transform_native_to_mni152_2mm ,nsd_betas_nii_to_numpy])
ds